# HMS hidden test set — late submission of the trained models

Scores our models on Kaggle's hidden test set (official labels we never saw). Every test EEG is a 50 s excerpt, so a 200 s model trained **with** the boundary shortcut always sees the "short recording" pattern here; a boundary-agnostic (BA) model should not be hurt by it.

**Inputs (Add Input), internet OFF:**
1. Competition: *HMS - Harmful Brain Activity Classification*
2. Output of the shortcut-check notebook (`notebookd13836b6e5`) — holds `v3ckpt/*_s1.pt` (50 s and original 200 s models)
3. Output of the v5 notebook (`notebookbb6726eb71`) — holds `results_v5/*.pt` and BA predictions
4. Output of the training notebook (`notebook86c832275b`, Version 5) — holds `results_v3/*_valpred.npy` (used to fit the temperature)

**How to submit:** set `SUBMIT` below → Save Version → Save & Run All (GPU T4) → open the version → *Submit to Competition*. One submission per model setting; Kaggle allows a few per day. The commit run first checks on 64 training EEGs that this inference path reproduces the saved out-of-fold predictions.

Suggested order (one submission each):
1. `mamba 200 ba_noreweight`  2. `mamba 200 s1` (original, shortcut)  3. `mamba 50 s1`  4. `gru 50 s1`  5. `transformer 50 s1`
6. `gru 200 s1`, `gru 200 ba`, `transformer 200 s1`, `transformer 200 ba`, `cnn 50 s1`

In [1]:
CFG = dict(
    # each entry: (model, context seconds, kind). kind = "s1" (original training, stage 1), "ba" or "ba_noreweight" (boundary-agnostic).
    # Several entries are averaged (e.g. seeds). Each entry uses all 5 fold models.
    SUBMIT=[("gru", 200, "ba_noreweight", 2042)],
    CALIBRATE=True,                      # temperature fitted on the model's pooled out-of-fold predictions
    # Label-prior checks (hidden test = 50 s excerpts only):
    # PRIOR_SHIFT="noctx" (or "ge10", "noctx_ge10"): multiply predictions by pi_noctx / pi_all and renormalise, where pi_noctx is the mean label of
    #   training EEGs whose 200 s window has < 10 s of extra real signal (the "short excerpt" EEGs) and pi_all the mean over all EEGs.
    # CONSTANT="all", "noctx", "ge10" or "noctx_ge10": ignore SUBMIT and predict that constant prior for every test EEG (no model, runs on CPU).
    PRIOR_SHIFT="ge10", CONSTANT=None,
    CKPT_GLOBS=["/kaggle/input/**/*.pt", "/kaggle/working/**/*.pt"],
    PRED_GLOBS=["/kaggle/input/**/*_valpred.npy"],
    N_VERIFY=64,                         # training EEGs used to check the inference path against saved predictions (0 = skip)
    BATCH=32, NUM_WORKERS=2, AMP=True,
    # architecture — must match training
    FS=100, PATCH=25, N_FOLDS=5, SEED=42,
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    DROPOUT=0.1, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False,
    T_GRID=[round(0.5 + 0.05 * i, 3) for i in range(51)],
    DATA_ROOT=None,
)

In [2]:
import os, glob, math, time, multiprocessing as mp
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MP = mp.get_context("fork")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/test.csv", "/kaggle/input/*/*/test.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "test_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")
ROOT = find_root()
print("data:", ROOT)

VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
P_COLS = [f"p_{c}" for c in CLASSES]

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Preprocessing (identical to training)

In [3]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"), ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"), ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),
           ("Fz", "Cz"), ("Cz", "Pz")]
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH, SRC_FS, CLIP_UV = len(BIPOLAR), 200, 500.0
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def make_view(x100, center_s, ctx, mode):
    """x100: whole recording at 100 Hz (T, 18). Window of ctx seconds centred on center_s, exactly as in training/OOF evaluation.
    mode "standard": zeros + mask 0 outside the recording.  mode "ba": mirror-filled, mask 1 everywhere."""
    fs = CFG["FS"]; n = int(ctx * fs); L = len(x100)
    i0 = int(round((center_s - n / fs / 2) * fs))
    a, b = max(i0, 0), min(i0 + n, L)
    if b <= a:
        return np.zeros((N_CH, n), np.float32), np.zeros(n, np.float32)
    if mode == "ba":
        lo, hi = a - i0, b - i0
        real = x100[a:b]
        x = np.pad(real, ((lo, n - hi), (0, 0)), mode="symmetric") if (lo > 0 or hi < n) else real
        m = np.ones(n, np.float32)
    else:
        x = np.zeros((n, N_CH), np.float32); m = np.zeros(n, np.float32)
        x[a - i0:b - i0] = x100[a:b]; m[a - i0:b - i0] = 1.0
    return np.ascontiguousarray(x.T / 50.0, dtype=np.float32), m

class ViewDataset(Dataset):
    def __init__(self, paths, centers, views):
        self.paths, self.centers, self.views = list(paths), list(centers), list(views)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        x100 = load_bipolar(self.paths[i])[0::2]
        out = []
        for ctx, mode in self.views:
            x, m = make_view(x100, self.centers[i], ctx, mode)
            out += [torch.from_numpy(x), torch.from_numpy(m)]
        return tuple(out)

## Models (identical to training)

In [4]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Checkpoints, out-of-fold predictions and folds

In [5]:
import pyarrow.parquet as pq

def find_files(globs):
    out = {}
    for pat in globs:
        for fp in sorted(glob.glob(pat, recursive=True)):
            out.setdefault(os.path.basename(fp), fp)
    return out
CKPTS, PREDS = find_files(CFG["CKPT_GLOBS"]), find_files(CFG["PRED_GLOBS"])
print(f"found {len(CKPTS)} checkpoints, {len(PREDS)} prediction files")

def entry_names(e, f):
    model, ctx, kind = e[:3]
    seed = e[3] if len(e) > 3 else 42
    if kind.startswith("ba"):
        tag = f"{model}_{kind}_ctx{ctx}_f{f}" + ("" if seed == 42 else f"_seed{seed}")
        return f"{tag}.pt", f"{tag}_ba_valpred.npy", "ba"
    return f"{model}_ctx{ctx}_f{f}_{kind}.pt", f"{model}_ctx{ctx}_f{f}_{kind}_valpred.npy", "standard"

ENTRIES = [] if CFG["CONSTANT"] else [tuple(e) for e in CFG["SUBMIT"]]
for e in ENTRIES:
    have = [f for f in range(CFG["N_FOLDS"]) if entry_names(e, f)[0] in CKPTS]
    print(f"  {e}: checkpoints for folds {have}")
    assert len(have) == CFG["N_FOLDS"], f"missing checkpoints for {e}: attach the notebook output that holds them"

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]
VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}

def temper_logp(logp, t):
    return torch.softmax(logp / t, -1)

def fit_temperature(e):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(entry_names(e, f)[1])
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), (fp, len(P), len(VA[f]))
        parts.append(P)
    P = np.concatenate(parts).astype(np.float64)
    Tt = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])])[P_COLS].to_numpy(np.float64)
    logp = np.log(np.clip(P, 1e-12, 1))
    def kl(t):
        L = logp / t; L -= L.max(1, keepdims=True); Q = np.exp(L); Q /= Q.sum(1, keepdims=True)
        return (Tt * (np.log(Tt + 1e-15) - np.log(Q + 1e-15))).sum(1).mean()
    t = min(CFG["T_GRID"], key=kl)
    print(f"  {e}: OOF KL raw {kl(1.0):.4f} -> calibrated {kl(t):.4f} at T = {t}")
    return t

def extra_context(center, dur, ctx=200):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(dur - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(dur - center, 25), 0, None)

PRIORS = {}
if CFG["PRIOR_SHIFT"] or CFG["CONSTANT"]:
    rows = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)
    t0 = time.time()
    dur = np.array([pq.ParquetFile(f"{ROOT}/train_eegs/{int(e)}.parquet").metadata.num_rows / SRC_FS for e in rows["eeg_id"]])
    nc = extra_context(rows["center"].to_numpy(), dur) < 10
    PRIORS = {"all": rows[P_COLS].to_numpy().mean(0), "noctx": rows.loc[nc, P_COLS].to_numpy().mean(0),
              "noctx_ge10": rows.loc[nc & (rows["n_votes"] >= 10), P_COLS].to_numpy().mean(0),
              "ge10": rows.loc[rows["n_votes"] >= 10, P_COLS].to_numpy().mean(0)}
    print(f"training EEGs: {len(rows)}, short-excerpt (no extra context): {nc.sum()} ({nc.mean():.1%}); "
          f"median votes all {rows['n_votes'].median():.0f}, short {rows.loc[nc, 'n_votes'].median():.0f}  [{time.time()-t0:.0f}s]")
    print(pd.DataFrame(PRIORS, index=CLASSES).T.round(3).to_string())

TEMPS = {}
for e in ENTRIES:
    t = fit_temperature(e) if CFG["CALIBRATE"] else None
    if CFG["CALIBRATE"] and t is None:
        print(f"  {e}: out-of-fold predictions not found -> no calibration (T = 1). Attach the output that holds them.")
    TEMPS[e] = t or 1.0

def load_model(e, f):
    m = Net(e[0]).to(DEV)
    m.load_state_dict(torch.load(CKPTS[entry_names(e, f)[0]], map_location=DEV))
    return m.eval()

def loader(paths, centers, views):
    return DataLoader(ViewDataset(paths, centers, views), CFG["BATCH"], shuffle=False, num_workers=CFG["NUM_WORKERS"],
                      multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)

@torch.no_grad()
def run(models, paths, centers):
    """models: list of (entry, model). Returns {entry: [per-model raw probabilities]} and the calibrated ensemble mean."""
    views = sorted({(e[1], entry_names(e, 0)[2]) for e, _ in models})
    vidx = {v: i for i, v in enumerate(views)}
    raw = {i: [] for i in range(len(models))}
    for batch in loader(paths, centers, views):
        for i, (e, m) in enumerate(models):
            j = vidx[(e[1], entry_names(e, 0)[2])]
            x, msk = batch[2 * j].to(DEV), batch[2 * j + 1].to(DEV)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
                raw[i].append(torch.log_softmax(m(x, msk).float(), -1).cpu())
    raw = {i: torch.cat(v) for i, v in raw.items()}
    ens = torch.stack([temper_logp(raw[i], TEMPS[e]) for i, (e, _) in enumerate(models)]).mean(0)
    return raw, ens.numpy().astype(np.float64)

found 125 checkpoints, 240 prediction files
  ('gru', 200, 'ba_noreweight', 2042): checkpoints for folds [0, 1, 2, 3, 4]
training EEGs: 17089, short-excerpt (no extra context): 9030 (52.8%); median votes all 3, short 2  [83s]
            Seizure    LPD    GPD   LRDA   GRDA  Other
all           0.154  0.142  0.104  0.065  0.115  0.419
noctx         0.064  0.135  0.121  0.043  0.095  0.541
noctx_ge10    0.043  0.136  0.122  0.048  0.075  0.576
ge10          0.042  0.183  0.125  0.074  0.090  0.485
  ('gru', 200, 'ba_noreweight', 2042): OOF KL raw 0.6901 -> calibrated 0.6858 at T = 1.1


## Check: this inference path reproduces the saved out-of-fold predictions
Fold-0 models score the first `N_VERIFY` fold-0 validation EEGs from the raw parquet files; the result must match the predictions saved during training (small differences from fp16 only).

In [6]:
if CFG["N_VERIFY"] > 0:
    rows = VA[0].head(CFG["N_VERIFY"])
    paths = [f"{ROOT}/train_eegs/{int(e)}.parquet" for e in rows["eeg_id"]]
    for e in ENTRIES:
        fp = PREDS.get(entry_names(e, 0)[1])
        if fp is None:
            print(f"  {e}: no saved fold-0 predictions, check skipped"); continue
        raw, _ = run([(e, load_model(e, 0))], paths, rows["center"].tolist())
        got = torch.softmax(raw[0], -1).numpy(); ref = np.load(fp)[:len(rows)]
        d = np.abs(got - ref).max()
        print(f"  {e}: max |p - saved p| over {len(rows)} EEGs = {d:.4f}")
        assert d < 0.05, f"{e}: inference does not reproduce the training-time predictions — do not submit, send me this log"
    print("verification passed")

  ('gru', 200, 'ba_noreweight', 2042): max |p - saved p| over 64 EEGs = 0.0005
verification passed


## Predict the test set and write `submission.csv`

In [7]:
test = pd.read_csv(f"{ROOT}/test.csv")
paths = [f"{ROOT}/test_eegs/{int(e)}.parquet" for e in test["eeg_id"]]
dur = np.array([pq.ParquetFile(p).metadata.num_rows / SRC_FS for p in paths])
centers = dur / 2                                     # test EEGs are 50 s excerpts labelled on their central 10 s
print(f"{len(test)} test EEGs, duration min/median/max = {dur.min():.1f}/{np.median(dur):.1f}/{dur.max():.1f} s")

t0 = time.time()
if CFG["CONSTANT"]:
    P = np.tile(PRIORS[CFG["CONSTANT"]], (len(test), 1))
    print(f"constant prediction = '{CFG['CONSTANT']}' training prior")
else:
    models = [(e, load_model(e, f)) for e in ENTRIES for f in range(CFG["N_FOLDS"])]
    _, P = run(models, paths, centers.tolist())
    print(f"predicted with {len(models)} models in {time.time()-t0:.0f}s; temperatures {TEMPS}")
    if CFG["PRIOR_SHIFT"]:
        P = P * (PRIORS[CFG["PRIOR_SHIFT"]] / PRIORS["all"])[None, :]
        P = P / P.sum(1, keepdims=True)
        print(f"prior shift applied: x pi_{CFG['PRIOR_SHIFT']} / pi_all")

sub = pd.read_csv(f"{ROOT}/sample_submission.csv")
cols = [c for c in sub.columns if c != "eeg_id"]
out = pd.DataFrame(P, columns=VOTES)
out.insert(0, "eeg_id", test["eeg_id"].to_numpy())
out = out[["eeg_id"] + cols]
assert np.allclose(out[cols].sum(axis=1), 1.0, atol=1e-4) and np.isfinite(out[cols].to_numpy()).all()
out.to_csv("/kaggle/working/submission.csv" if os.path.isdir("/kaggle/working") else "submission.csv", index=False)
print("submission written for:", ENTRIES or f"constant {CFG['CONSTANT']}", "| calibrated:", CFG["CALIBRATE"], "| prior shift:", CFG["PRIOR_SHIFT"])
print(out.head())

1 test EEGs, duration min/median/max = 50.0/50.0/50.0 s
predicted with 5 models in 1s; temperatures {('gru', 200, 'ba_noreweight', 2042): 1.1}
prior shift applied: x pi_ge10 / pi_all
submission written for: [('gru', 200, 'ba_noreweight', 2042)] | calibrated: True | prior shift: ge10
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.018033  0.033816  0.003165   0.118369   0.055195   

   other_vote  
0    0.771422  
